In [0]:
from delta.tables import DeltaTable
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, DoubleType, TimestampType

# Replace this with your own target table when you want to run the load
target_table = "catalog.schema.customer_target"
key_columns = ["id"]
watermark_column = "updated_at"


schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("updated_at", TimestampType(), True),
])


full_source_data = [
    (1, "Asha", "Bangalore", 1200.0, datetime.strptime("2026-06-01 09:00:00", "%Y-%m-%d %H:%M:%S")), 
    (2, "Rahul", "Hyderabad", 950.0, datetime.strptime("2026-06-01 10:30:00", "%Y-%m-%d %H:%M:%S")), 
    (3, "Meera", "Chennai", 1500.0, datetime.strptime("2026-06-01 11:15:00", "%Y-%m-%d %H:%M:%S")), 
    (4, "John", "Pune", 800.0, datetime.strptime("2026-06-01 12:00:00", "%Y-%m-%d %H:%M:%S")), 
]

incremental_source_data = [
    (2, "Rahul", "Hyderabad", 1100.0, datetime.strptime("2026-06-02 08:00:00", "%Y-%m-%d %H:%M:%S")), 
    (3, "Meera", "Chennai", 1700.0, datetime.strptime("2026-06-02 09:00:00", "%Y-%m-%d %H:%M:%S")), 
    (5, "Sara", "Mumbai", 1400.0, datetime.strptime("2026-06-02 10:00:00", "%Y-%m-%d %H:%M:%S")), 
    (6, "David", "Delhi", 1250.0, datetime.strptime("2026-06-02 11:00:00", "%Y-%m-%d %H:%M:%S"))
]


full_source_df = spark.createDataFrame(full_source_data, schema=schema) \
    .selectExpr(
        "id",
        "customer_name",
        "city",
        "amount",
        "cast(updated_at as timestamp) as updated_at"
    )

incremental_source_df = spark.createDataFrame(incremental_source_data, schema=schema) \
    .selectExpr(
        "id",
        "customer_name",
        "city",
        "amount",
        "cast(updated_at as timestamp) as updated_at"
    )

# Temp views for the SQL cell
full_source_df.createOrReplaceTempView("source_full_df_view")
incremental_source_df.createOrReplaceTempView("source_incremental_df_view")


def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name)
        return True
    except Exception:
        return False


def full_load_df(source_df, target_table: str):
    (
        source_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"Full load completed into {target_table}")
    display(spark.table(target_table).limit(5))


def incremental_load_df(source_df, target_table: str, key_columns: list, watermark_column: str):
    if not table_exists(target_table):
        print(f"Target {target_table} does not exist. Running initial full load instead.")
        (
            source_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )
        display(spark.table(target_table).limit(5))
        return

    delta_target = DeltaTable.forName(spark, target_table)
    merge_condition = " AND ".join([f"t.{c} = s.{c}" for c in key_columns])
    update_condition = f"s.{watermark_column} >= t.{watermark_column}"

    (
        delta_target.alias("t")
        .merge(source_df.alias("s"), merge_condition)
        .whenMatchedUpdateAll(condition=update_condition)
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Incremental load completed into {target_table}")
    display(spark.table(target_table).orderBy("id").limit(10))


print("Full load source dataframe")
display(full_source_df)

print("Incremental load source dataframe")
display(incremental_source_df)

# Example usage after replacing target_table with a real table name
# full_load_df(full_source_df, target_table)
# incremental_load_df(incremental_source_df, target_table, key_columns, watermark_column)


In [0]:
%sql
-- Replace target_table below with your own table before running

-- Example source dataframes are created in the Python cell as temp views:
-- source_full_df_view
-- source_incremental_df_view

-- FULL LOAD
CREATE OR REPLACE TABLE catalog.schema.customer_target AS
SELECT *
FROM source_full_df_view;

SELECT *
FROM catalog.schema.customer_target
ORDER BY id;

-- INCREMENTAL LOAD
MERGE INTO catalog.schema.customer_target AS t
USING source_incremental_df_view AS s
ON t.id = s.id
WHEN MATCHED AND s.updated_at >= t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT *
FROM catalog.schema.customer_target
ORDER BY id;
